# CodeBERT experiments on Kaggle

Before running, in the right-hand **Session options** panel:
- **Accelerator:** GPU T4 x2 (or P100)
- **Internet:** On (needed to clone the public repo and download `microsoft/codebert-base`)

Run the setup and GPU-check cells first. Then **Save Version -> Save & Run All (Commit)**.
When it finishes, download `results.zip` from the **Output** tab and unzip it into `experiments/`
in the repo. Each archive contains a new `kaggle_<timestamp>/` folder, preserving existing Mac results.

Runs CodeBERT LoRA and TF-IDF + LinearSVC (standard and balanced) on the control, modules and newlines variants.
Comments and combined-fix datasets are excluded. Qwen requires a separate CUDA port.


In [ ]:
import shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/alexsmyr0/thesis-experiments"
REPO = Path("/tmp/thesis-experiments")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO)], check=True)
# Use the notebook's interpreter and install every pipeline dependency.
# Stop immediately if setup fails. Existing compatible packages (including CUDA torch) stay installed.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", COMMIT)


In [ ]:
# Check in a fresh process, matching the interpreter used by the training scripts.
# This avoids stale imports if Kaggle loaded a package before pip updated it.
import json

probe = subprocess.check_output([sys.executable, "-c", """
import importlib.metadata as metadata
import json
import torch
import transformers, peft, datasets, accelerate, sklearn, pandas, matplotlib, seaborn
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU in Session options, then restart"
from transformers import TrainingArguments
assert TrainingArguments(output_dir='/tmp/kaggle-preflight', warmup_steps=0.1, report_to='none').get_warmup_steps(100) == 10
packages = ['torch', 'transformers', 'peft', 'datasets', 'accelerate',
            'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn']
print(json.dumps({
    'versions': {name: metadata.version(name) for name in packages},
    'cuda': torch.version.cuda,
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}))
"""], text=True)
RUNTIME = json.loads(probe)
print(json.dumps(RUNTIME, indent=2))


In [ ]:
import os
from datetime import datetime, timezone

# Run every variant with the same hardware and settings, including the control.
DATASETS = ["control_unmodified", "only_modules", "only_newlines"]
CODEBERT = "lora"      # "full", "lora" or "both"
TFIDF = "svm"          # LinearSVC, standard and balanced class weights
FP16 = False           # Opt-in: faster on T4, but can shift scores

RUN_NAME = datetime.now(timezone.utc).strftime("kaggle_%Y%m%dT%H%M%S_%fZ")
OUT = Path("/kaggle/working/experiments") / RUN_NAME
OUT.mkdir(parents=True, exist_ok=False)
(OUT / "run_metadata.json").write_text(json.dumps({
    'run': RUN_NAME, 'repo': REPO_URL, 'commit': COMMIT,
    'datasets': DATASETS, 'codebert': CODEBERT, 'tfidf': TFIDF, 'fp16': FP16, **RUNTIME,
}, indent=2))
(OUT / "pip_freeze.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))

for name in DATASETS:
    print(f"\n######## {name} ########", flush=True)
    env = {**os.environ,
           "THESIS_CSV": str(REPO / "data" / "v2" / f"{name}.csv"),
           "THESIS_OUT": str(OUT / name),
           "PY": sys.executable,
           "CODEBERT_ARGS": "--fp16" if FP16 else ""}
    subprocess.run(["sh", str(REPO / "scripts" / "run_all.sh"), f"--codebert={CODEBERT}", f"--tfidf={TFIDF}"], env=env, check=True)
    shutil.rmtree(OUT / name / "checkpoints", ignore_errors=True)  # keep the Output small


In [ ]:
for name in DATASETS:
    print(f"\n=== {name} ===")
    print((OUT / name / "logs" / "summary.txt").read_text())
shutil.make_archive("/kaggle/working/results", "zip", OUT.parent, OUT.name)
print(f"Wrote /kaggle/working/results.zip (contains {RUN_NAME}/)")
